# 07 · Texturas de documentos narrativos — papel, tinta y envejecimiento

**V2:** exporta también fondos WebP optimizados a 960×540 para evitar PNG de 1.4 MB en Android. Los demos nunca son producción.

**Objetivo:** crear overlays transparentes reutilizables de textura victoriana para cartas, expedientes y recortes, sin tocar el HTML de los documentos existentes ni introducir texto generado erróneo. No se depende de un asset concreto: los textos del juego siguen controlados por el motor.

Todo se genera de forma determinista con Pillow + ruido y semillas, **sin GPU ni APIs de pago**. Los PNG transparentes **no deben integrarse automáticamente**: primero elegir dónde aplicarlos por CSS o Canvas sin reducir la legibilidad de los textos. Se exporta tanto un mockup como capas individuales para composición.

In [ ]:
%pip -q install 'Pillow>=10' 'numpy>=1.24' 'requests>=2.31'

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance, ImageOps, ImageChops
from IPython.display import display
import base64, io, json, math, hashlib, random, requests, shutil
REPO = "gabsvm/HorrorPlay"
COMMIT = "0681cfee36921ec2bd372f8ae8080a401f8bd63b"

def source(path):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    r = requests.get(url, params={"ref": COMMIT}, headers={"Accept":"application/vnd.github+json"}, timeout=90)
    r.raise_for_status()
    data = r.json()
    if data.get("encoding") != "base64":
        raise RuntimeError(f"GitHub no devolvió el JPG/PNG en base64: {path}")
    b = base64.b64decode(data["content"])
    result = Image.open(io.BytesIO(b)).convert("RGB" if path.lower().endswith((".jpg", ".jpeg")) else "RGBA")
    return result, hashlib.sha256(b).hexdigest()

def font(size=20, bold=False):
    files = ["/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
             "/usr/share/fonts/truetype/liberation2/LiberationSerif-Regular.ttf"]
    for f in files:
        if Path(f).exists(): return ImageFont.truetype(f, size)
    return ImageFont.load_default()

def compare(images, thumb=(640,360), labels=None):
    w,h=thumb
    sheet=Image.new('RGB',(w*len(images),h+36),(17,18,20))
    dr=ImageDraw.Draw(sheet)
    for i,im in enumerate(images):
        sheet.paste(im.convert('RGB').resize((w,h), Image.Resampling.LANCZOS),(i*w,36))
        dr.text((i*w+12,10), labels[i] if labels else str(i),fill=(245,241,229),font=font(15))
    return sheet

def finish(folder, manifest, outputs):
    manifest.update({"repo":REPO,"source_commit":COMMIT,"status":"CANDIDATO - No integrado ni testeado en Android"})
    (folder/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    archive=shutil.make_archive(str(folder),'zip',folder)
    print('Exportado:', archive, 'archivos:', len(list(folder.iterdir())))
    from google.colab import files
    files.download(archive)

In [ ]:
import numpy as np
out=Path('/content/horrorplay_document_textures');out.mkdir(exist_ok=True,parents=True)
W,H=1280,720

def paper(seed,style):
    r=np.random.default_rng(seed)
    yy,xx=np.mgrid[0:H,0:W]
    edge=np.minimum.reduce([xx,yy,W-1-xx,H-1-yy]).astype('float32')
    age=np.exp(-edge/65.)
    noise=r.normal(0,5.5,(H,W)).astype('float32')
    stains=np.zeros((H,W),dtype='float32')
    for _ in range(15 if style=='letter' else 28):
        cx=int(r.integers(0,W));cy=int(r.integers(0,H))
        sx=int(r.integers(25,135));sy=int(r.integers(10,110))
        stains+=r.uniform(.015,.08)*np.exp(-(((xx-cx)/sx)**2+((yy-cy)/sy)**2)*2.0)
    base=np.zeros((H,W,3),dtype='float32')
    parchment=(216,195,157) if style=='letter' else (184,166,138)
    for channel,v in enumerate(parchment):base[...,channel]=v+noise-age*(24+channel*6)-stains*70
    return Image.fromarray(np.uint8(np.clip(base,0,255)),'RGB')

def overlay(seed,style):
    paper_im=paper(seed,style)
    # Subtle grayscale texture alpha. Intended as layer, not ink/text.
    a=np.asarray(paper_im.convert('L'),dtype=np.float32)
    opacity=np.uint8(np.clip((220-a)*.23,0,42))
    rgba=Image.new('RGBA',(W,H),(65,43,22,0));rgba.putalpha(Image.fromarray(opacity,'L'))
    return rgba

styles=[('carta',1888,'letter'),('expediente',1890,'dossier'),('recorte',1892,'news')]
for name,seed,style in styles:
    bg=paper(seed,style)
    alpha=overlay(seed,style)
    bg.save(out/f'{name}_paper_1280x720.png')
    alpha.save(out/f'{name}_overlay_transparente.png')
    # Asset ligero listo para WebView; el original PNG queda como referencia.
    bg.resize((960,540),Image.Resampling.LANCZOS).save(out/f'{name}_paper_960x540.webp',format='WEBP',quality=74,method=6)

    # Mockup: el texto es una etiqueta ficticia para evaluar CONTRASTE,
    # jamás contenido narrativo a integrar.
    mock=bg.copy();d=ImageDraw.Draw(mock)
    d.rectangle((160,95,1119,620),outline=(102,76,51,100),width=2)
    d.text((205,160),'MUESTRA DE PAPEL  ·  SOLO PRUEBA',font=font(34,True),fill=(53,41,30))
    d.text((205,235),'Los textos narrativos permanecerán en el motor.',font=font(22),fill=(72,55,40))
    d.text((205,284),'Verifica la lectura a 640 × 360.',font=font(22),fill=(72,55,40))
    mock.save(out/f'{name}_demo_1280x720.png')
examples=[Image.open(out/f'{n}_demo_1280x720.png') for n,_,_ in styles]
sheet=compare(examples,(640,360),['Carta','Expediente','Recorte'])
sheet.save(out/'AB_papeles_640.png');display(sheet)
finish(out,{'seeds':[1888,1890,1892], 'output_type':'Capas de papel; mockups NO canon',
 'warning':'NO incluir los mockups en producción; preservar HTML/CSS/textos y accesibilidad'},[])